# Kombinierter LSTM: Kapazitäts-Screening

## Fragestellung

Der kombinierte LSTM mit AEZ- und Monatsembeddings war die beste sequenzielle Variante. Dieses Notebook prüft ausschließlich auf den fünf Trainingsfolds, ob mehr Encoderbreite, zusätzliche LSTM-Schichten oder eine höhere Epochenobergrenze die kombinierte Balanced Accuracy verbessern.

Der gemeinsame Holdout wurde für die LSTM-Familie bereits ausgewertet. Deshalb enthält dieses Notebook bewusst keinen `run()`-Aufruf und erzeugt keine neue Holdout-Metrik.

## Versuchsdesign

Das Raster enthält acht Kandidaten: 64 oder 128 Hidden Units, eine oder zwei LSTM-Schichten sowie 100 oder 200 Epochen als Obergrenze. Early Stopping bleibt unverändert aktiv; mehr Epochen verlängern nur den maximal möglichen Lauf.

AEZ und Month bleiben lernbare Embeddings. Vorverarbeitung, Split, Folds und primäre Metrik bleiben gegenüber dem bisherigen Embedding-LSTM unverändert.

In [ ]:
import importlib.metadata
import platform
import subprocess

import numpy as np
import torch
from IPython.display import display

from awp2.config import LSTM_EMBEDDINGS_CONTEXT, SEED
from awp2.experiment import TuneConfig, tune
from awp2.models import make_combined_spectral_lstm
from awp2.preprocessing import PreprocessingConfig

EXPERIMENT_ID = "combined_lstm_capacity"
TUNING_ID = f"{EXPERIMENT_ID}_search"
PREPROCESSING = PreprocessingConfig(
    use_spectral_standard_scale=True,
    use_categorical_context=True,
)
PARAM_GRID = {
    "estimator__hidden_size": [64, 128],
    "estimator__num_layers": [1, 2],
    "estimator__max_epochs": [100, 200],
}

np.random.seed(SEED)
torch.manual_seed(SEED)
environment = {
    "python": platform.python_version(),
    "torch": importlib.metadata.version("torch"),
    "scikit_learn": importlib.metadata.version("scikit-learn"),
    "branch": subprocess.check_output(
        ["git", "branch", "--show-current"],
        text=True,
    ).strip(),
    "preprocessing": PREPROCESSING.model_dump(),
    "param_grid": PARAM_GRID,
}
display(environment)

tuned = tune(
    make_combined_spectral_lstm(LSTM_EMBEDDINGS_CONTEXT),
    TuneConfig(
        name=TUNING_ID,
        approach="combined",
        description=(
            "CV-only: Kapazitäts-Screening für den kombinierten LSTM "
            "mit AEZ- und Monatsembeddings."
        ),
        preprocessing=PREPROCESSING,
        param_grid=PARAM_GRID,
    ),
)
results = tuned.candidates.sort_values("rank")
display(results)
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## Auswertung

Die beste Konfiguration verwendet `hidden_size=64`, `num_layers=2` und `max_epochs=200`. Sie erreicht eine mittlere CV-BAcc kombiniert von `0.73258` bei einer Fold-Standardabweichung von `0.02341`. Gegenüber dem bisherigen kombinierten Embedding-LSTM (`0.71072`) beträgt der Gewinn `0.02186` und liegt damit klar über `MIN_CV_GAIN = 0.005`.

Zwei LSTM-Schichten helfen bei 64 Hidden Units: Die besten einlagigen Kandidaten liegen bei `0.71904`, der zweilagige Kandidat mit maximal 100 Epochen bei `0.73134`. Die Erhöhung von 100 auf 200 Epochen bringt dagegen nur `0.00124` und liegt unter der Entscheidungsschwelle; Early Stopping begrenzt die zusätzliche Laufzeit. 128 Hidden Units verschlechtern die Ergebnisse auf maximal `0.72398` und erhöhen zugleich die Fold-Streuung.

Für die weitere CV-only-Entwicklung bleibt daher die kompakte zweilagige Architektur mit 64 Hidden Units die Referenz. Auch bei einem klaren CV-Gewinn wird kein weiterer Holdout-Lauf gestartet: Der gemeinsame Holdout wurde für die LSTM-Familie bereits verwendet. Die nächste getrennte Hypothese ist ein BiLSTM mit derselben CV-only-Regel.

MLflow konnte diesen Lauf wegen `Permission denied: '/C:'` nicht protokollieren; deshalb ist keine Run-ID verfügbar.